# Salary Model — AI Career Advisor

Trains and evaluates salary regression models, saves the best to `models/salary_model.pkl`.

In [ ]:
import os, sys, warnings
sys.path.insert(0, os.path.abspath('..'))
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
try:
    from xgboost import XGBRegressor
    HAS_XGB = True
except ImportError:
    HAS_XGB = False
    print('XGBoost not available, skipping.')

os.makedirs('../models', exist_ok=True)
salary_df = pd.read_csv('../data/salary.csv')
print(f'Loaded {salary_df.shape[0]} rows')
salary_df.head()

In [ ]:
# ── Preprocessing ──────────────────────────────────────────────────────────
CATEGORICAL = ['Education', 'Location', 'Job Title', 'Company Size', 'Employment Type']
NUMERIC     = ['Experience']
TARGET      = 'Salary'

# Skill TF-IDF
skill_tfidf = TfidfVectorizer(max_features=50)
skill_tfidf.fit(salary_df['Skills'].fillna(''))
skill_matrix = skill_tfidf.transform(salary_df['Skills'].fillna('')).toarray()
skill_cols   = [f'skill_{w}' for w in skill_tfidf.get_feature_names_out()]

skill_feat_df = pd.DataFrame(skill_matrix, columns=skill_cols)
feat_df = pd.concat([
    salary_df[CATEGORICAL + NUMERIC + [TARGET]].reset_index(drop=True),
    skill_feat_df.reset_index(drop=True)
], axis=1)

# Encode
cat_enc    = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
num_scaler = StandardScaler()
cat_features  = cat_enc.fit_transform(feat_df[CATEGORICAL])
num_features  = num_scaler.fit_transform(feat_df[NUMERIC])
skill_features = feat_df[skill_cols].values

X = np.hstack([num_features, cat_features, skill_features])
y = feat_df[TARGET].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f'Train: {X_train.shape}  |  Test: {X_test.shape}')

In [ ]:
# ── Train & Evaluate ───────────────────────────────────────────────────────
models_to_train = {
    'Linear Regression': LinearRegression(),
    'Decision Tree':     DecisionTreeRegressor(max_depth=10, random_state=42),
    'Random Forest':     RandomForestRegressor(n_estimators=150, max_depth=15, random_state=42, n_jobs=-1),
}
if HAS_XGB:
    models_to_train['XGBoost'] = XGBRegressor(
        n_estimators=150, learning_rate=0.1, max_depth=6, random_state=42,
        verbosity=0, n_jobs=-1)

results = {}
best_r2, best_name, best_model = -np.inf, None, None

print(f"{'Model':<25} {'MAE':>10} {'MSE':>14} {'RMSE':>12} {'R²':>8}")
print('-' * 72)
for name, mdl in models_to_train.items():
    mdl.fit(X_train, y_train)
    preds = mdl.predict(X_test)
    mae  = mean_absolute_error(y_test, preds)
    mse  = mean_squared_error(y_test, preds)
    rmse = np.sqrt(mse)
    r2   = r2_score(y_test, preds)
    results[name] = {'MAE': mae, 'MSE': mse, 'RMSE': rmse, 'R²': r2}
    print(f"{name:<25} {mae:>10,.0f} {mse:>14,.0f} {rmse:>12,.0f} {r2:>8.4f}")
    if r2 > best_r2:
        best_r2, best_name, best_model = r2, name, mdl

print(f'\n✅ Best model: {best_name}  R²={best_r2:.4f}')

In [ ]:
# ── Performance Visualization ──────────────────────────────────────────────
preds_best = best_model.predict(X_test)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].scatter(y_test, preds_best, alpha=0.4, color='#4f86c6', s=15)
line = np.linspace(y_test.min(), y_test.max(), 100)
axes[0].plot(line, line, 'r--', linewidth=2)
axes[0].set_title(f'{best_name} — Predicted vs Actual', fontweight='bold')
axes[0].set_xlabel('Actual Salary'); axes[0].set_ylabel('Predicted Salary')

residuals = y_test - preds_best
axes[1].hist(residuals, bins=40, color='#e06c75', edgecolor='white', alpha=0.8)
axes[1].axvline(0, color='white', linestyle='--')
axes[1].set_title('Residuals Distribution', fontweight='bold')
axes[1].set_xlabel('Residual (USD)')
plt.tight_layout()
plt.savefig('../images/salary_model_performance.png', dpi=120)
plt.show()

# Bar comparison
res_df = pd.DataFrame(results).T
res_df[['R²']].plot(kind='bar', figsize=(8, 4), color='#56b6c2', legend=False)
plt.title('R² Score Comparison', fontweight='bold')
plt.xticks(rotation=15)
plt.ylim(0, 1.05)
plt.tight_layout()
plt.savefig('../images/model_r2_comparison.png', dpi=120)
plt.show()

In [ ]:
# ── Feature Importance (Random Forest / XGBoost) ──────────────────────────
if hasattr(best_model, 'feature_importances_'):
    # Category names
    cat_names = cat_enc.get_feature_names_out(CATEGORICAL).tolist()
    num_names = NUMERIC
    all_names = num_names + cat_names + skill_cols
    importances = best_model.feature_importances_
    feat_imp = pd.Series(importances, index=all_names).sort_values(ascending=False).head(20)
    fig, ax = plt.subplots(figsize=(10, 5))
    feat_imp[::-1].plot(kind='barh', ax=ax, color='#c678dd')
    ax.set_title('Top 20 Feature Importances', fontweight='bold')
    plt.tight_layout()
    plt.savefig('../images/feature_importance.png', dpi=120)
    plt.show()

In [ ]:
# ── Save Best Model ────────────────────────────────────────────────────────
salary_bundle = {
    'model':            best_model,
    'skill_tfidf':      skill_tfidf,
    'cat_enc':          cat_enc,
    'num_scaler':       num_scaler,
    'skill_cols':       skill_cols,
    'categorical':      CATEGORICAL,
    'numeric':          NUMERIC,
    'results':          results,
    'best_model_name':  best_name,
}
joblib.dump(salary_bundle, '../models/salary_model.pkl')
print('✅ Saved → ../models/salary_model.pkl')

# Quick smoke test
loaded = joblib.load('../models/salary_model.pkl')
m = loaded['model']; sc = loaded['skill_tfidf']; ce = loaded['cat_enc']; ns = loaded['num_scaler']
test_row = pd.DataFrame([{'Experience': 5, 'Education': 'Master', 'Location': 'New York',
                           'Job Title': 'Data Scientist', 'Company Size': 'Large', 'Employment Type': 'Full-time'}])
cf = ce.transform(test_row[CATEGORICAL])
nf = ns.transform(test_row[['Experience']])
sf = sc.transform(['Python,Machine Learning,SQL']).toarray()
X_t = np.hstack([nf, cf, sf])
print(f'Sample prediction: ${m.predict(X_t)[0]:,.0f}')